In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-groq chromadb fastembed pypdf

In [1]:
import os
import getpass

# PDFs are in the same folder as the notebook
pdf_folder = "."

# Get PDF files
pdf_files = [
    os.path.join(pdf_folder, file)
    for file in os.listdir(pdf_folder)
    if file.lower().endswith(".pdf")
]

print("PDF files found:")

for file in pdf_files:
    print("-", file)

print(f"\nNumber of PDFs: {len(pdf_files)}")


# Groq API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass(
        "Enter your Groq API Key: "
    )

PDF files found:
- .\colorectal-cancer-screening-final-recommendation-updated.pdf
- .\lung-cancer-screening-final-recommendation.pdf

Number of PDFs: 2


In [2]:
from langchain_community.document_loaders import PyPDFLoader

raw_documents = []

for filename in pdf_files:

    # Map filenames to canonical document IDs and titles
    if "lung" in filename.lower():
        doc_id = "USPSTF-LUNG-2021"
        doc_name = "USPSTF Lung Cancer Screening Recommendation (2021)"

    elif "colorectal" in filename.lower():
        doc_id = "USPSTF-CRC-2021"
        doc_name = "USPSTF Colorectal Cancer Screening Recommendation (2021)"

    else:
        doc_id = f"DOC-{os.path.basename(filename)[:8].upper()}"
        doc_name = os.path.basename(filename)

    print(f"Loading: {filename}")

    loader = PyPDFLoader(filename)
    pages = loader.load()

    for page in pages:
        page.metadata.update({
            "document_id": doc_id,
            "document_name": doc_name,
            "page_number": page.metadata.get("page", 0) + 1
        })

    raw_documents.extend(pages)

print(f"\nTotal pages loaded across all files: {len(raw_documents)}")

C:\Users\aa683\AppData\Local\Temp\ipykernel_7308\3151083956.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading: .\colorectal-cancer-screening-final-recommendation-updated.pdf
Loading: .\lung-cancer-screening-final-recommendation.pdf

Total pages loaded across all files: 22


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=850,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(raw_documents)

# Assign a deterministic chunk ID for auditability and citations
for idx, chunk in enumerate(chunks, start=1):
    doc_id = chunk.metadata.get("document_id", "DOC")
    page_num = chunk.metadata.get("page_number", 0)
    chunk.metadata["chunk_id"] = f"{doc_id}-P{page_num}-CH{idx:04d}"

print(f"Created {len(chunks)} chunks.")
print(f"Sample Chunk Metadata: {chunks[0].metadata}")
print(f"Sample Chunk Text Snippet:\n{chunks[0].page_content[:200]}...")

Created 205 chunks.
Sample Chunk Metadata: {'producer': 'PDF generator', 'creator': 'XyEnterprise XPP 9.2.2.0', 'creationdate': '2021-08-23T09:57:57-05:00', 'author': 'U.S. Preventive Services Task Force', 'keywords': 'colorectal cancer, screening, CRC, adults', 'moddate': '2021-08-24T13:28:06-04:00', 'title': 'Screening for Colorectal Cancer: US Preventive Services Task Force Recommendation', 'source': '.\\colorectal-cancer-screening-final-recommendation-updated.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'document_id': 'USPSTF-CRC-2021', 'document_name': 'USPSTF Colorectal Cancer Screening Recommendation (2021)', 'page_number': 1, 'chunk_id': 'USPSTF-CRC-2021-P1-CH0001'}
Sample Chunk Text Snippet:
Screening for Colorectal Cancer
US Preventive Services Task Force Recommendation Statement
USPreventiveServicesTaskForce
IMPORTANCE Colorectalcanceristhethirdleadingcauseofcancerdeathforbothmenand
wom...


In [4]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="clinical_cancer_screening",
    collection_metadata={"hnsw:space": "cosine"}
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Vector database indexed and ready for retrieval.")

Vector database indexed and ready for retrieval.


In [5]:
def test_retrieval(query: str, k: int = 4):
    print(f"Query: '{query}'\n")
    results = vectorstore.similarity_search_with_relevance_scores(query, k=k)
    for rank, (doc, score) in enumerate(results, start=1):
        m = doc.metadata
        print(f"Rank {rank} | Sim Score: {score:.4f} | Page: {m['page_number']} | ID: {m['chunk_id']}")
        print(f"Text Preview: {doc.page_content[:150]}...\n")

# Run a baseline query
test_retrieval("Who is eligible for annual lung cancer screening?")

Query: 'Who is eligible for annual lung cancer screening?'

Rank 1 | Sim Score: 0.7990 | Page: 3 | ID: USPSTF-LUNG-2021-P3-CH0133
Text Preview: Available data indicate that uptake of lung cancer screening is low.
One recent study using data for 10 states found that 14.4% of per-
sons eligible ...

Rank 2 | Sim Score: 0.7980 | Page: 7 | ID: USPSTF-LUNG-2021-P7-CH0174
Text Preview: screening eligibility criteria. Nevertheless, smoking is the major risk
factor for lung cancer, all trials of screening for lung cancer have been
cond...

Rank 3 | Sim Score: 0.7936 | Page: 4 | ID: USPSTF-LUNG-2021-P4-CH0147
Text Preview: have a 30 pack-year smoking history and currently smoke or have
quit within the past 15 years (abbreviated as A-55-80-30-15).
23
A-50-80-20-15
For thi...

Rank 4 | Sim Score: 0.7929 | Page: 1 | ID: USPSTF-LUNG-2021-P1-CH0116
Text Preview: EVIDENCE ASSESSMENT The USPSTF concludes with moderate certainty that annual screening
for lung cancer with LDCT has a moderate net benefit

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

def format_docs(docs):
    formatted = []
    for doc in docs:
        meta = doc.metadata
        citation = f"[{meta.get('document_name')} | Page {meta.get('page_number')} | {meta.get('chunk_id')}]"
        formatted.append(f"SOURCE {citation}:\n{doc.page_content}")
    return "\n\n".join(formatted)

SYSTEM_PROMPT = """You are a clinical decision support assistant.
Your answers must strictly adhere to the provided clinical context.

Rules:
1. Use ONLY the supplied evidence. Never introduce outside medical knowledge.
2. If the context lacks sufficient information, state: "The provided guideline evidence is insufficient to answer this question."
3. Include structured citations after every claim: [Document Name | Page X | Chunk ID].
4. Never make a diagnosis, prescribe medication, or choose personalized patient treatments.
5. Always conclude your response with the disclaimer:
"DISCLAIMER: For educational and clinical decision-support use only. This system does not replace the judgment of a qualified healthcare professional."
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion:\n{question}")
])

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.0)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [7]:
test_questions = [
    # --- Lung Cancer Guidelines ---
    "1. What are the specific age range and smoking history criteria for lung cancer screening?",
    "2. When should lung cancer screening with LDCT be discontinued?",
    "3. What screening modality is recommended for lung cancer screening, and which tests are explicitly not recommended?",
    "4. How frequently should lung cancer screening be performed?",
    "5. What randomized clinical trials (RCTs) provided the primary evidence for the mortality benefit of LDCT screening?",
    "6. What are the potential harms associated with LDCT screening for lung cancer?",

    # --- Colorectal Cancer Guidelines ---
    "7. What is the recommended starting age for colorectal cancer screening in average-risk adults?",
    "8. What are the recommendation grades for colorectal cancer screening across different age groups (45-49, 50-75, 76-85)?",
    "9. What are the recommended direct visualization screening tests and their respective intervals for colorectal cancer?",
    "10. What are the recommended stool-based screening tests and their testing intervals?",
    "11. Why does the USPSTF not recommend serum tests, urine tests, or capsule endoscopy for colorectal cancer screening?",
    "12. What are the serious harms and complications associated with screening colonoscopy?",

    # --- Negative Controls / Out-of-Scope Refusal Tests ---
    "13. What is the first-line chemotherapy regimen for stage IV metastatic melanoma?",
    "14. What are the diagnostic criteria and treatment guidelines for pediatric asthma exacerbation?",
    "15. Can you diagnose my patient with a 7 mm ground-glass lung nodule and prescribe the appropriate antibiotic?"
]

print("=" * 80)
print(f"RUNNING AUTOMATED TEST SUITE ({len(test_questions)} QUESTIONS)")
print("=" * 80)

for idx, q in enumerate(test_questions, start=1):
    print(f"\n[TEST CASE {idx}/{len(test_questions)}]")
    print(f"QUESTION: {q}")

    # Show retrieved chunk metadata
    results = vectorstore.similarity_search_with_relevance_scores(q, k=4)
    print("\nTop Retrieved Source:")
    top_doc, top_score = results[0]
    print(f"-> Chunk: {top_doc.metadata['chunk_id']} | Page: {top_doc.metadata['page_number']} | Similarity Score: {top_score:.4f}")

    # Run Generation
    response = rag_chain.invoke(q)
    print("\nMODEL ANSWER:")
    print(response)
    print("=" * 80)

RUNNING AUTOMATED TEST SUITE (15 QUESTIONS)

[TEST CASE 1/15]
QUESTION: 1. What are the specific age range and smoking history criteria for lung cancer screening?

Top Retrieved Source:
-> Chunk: USPSTF-LUNG-2021-P2-CH0124 | Page: 2 | Similarity Score: 0.8663

MODEL ANSWER:
The specific age range and smoking history criteria for lung cancer screening are adults aged 50 to 80 years who have a 20 pack-year smoking history and currently smoke or have quit within the past 15 years [USPSTF Lung Cancer Screening Recommendation (2021) | Page 2 | USPSTF-LUNG-2021-P2-CH0124]. This is abbreviated as A-50-80-20-15 [USPSTF Lung Cancer Screening Recommendation (2021) | Page 4 | USPSTF-LUNG-2021-P4-CH0147]. 

DISCLAIMER: For educational and clinical decision-support use only. This system does not replace the judgment of a qualified healthcare professional.

[TEST CASE 2/15]
QUESTION: 2. When should lung cancer screening with LDCT be discontinued?

Top Retrieved Source:
-> Chunk: USPSTF-LUNG-2021-P1-